# Day 5-02｜建立成果圖：角度圖、球軌跡、骨架示意

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：把結果整理成展示圖片。學生可以挑一張圖講：我的投籃動作有什麼特徵？AI 分析有什麼限制？

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
import pandas as pd
from src.shooting_utils import synthetic_pose_sequence, draw_skeleton
from src.plot_utils import plot_angle_series, plot_ball_path
from src.cv_utils import save_image_rgb, show_image

RESULTS = COURSE_ROOT / "assets" / "results"
pose_csv = RESULTS / "d4_02_pose_angles.csv"
ball_csv = RESULTS / "d4_03_ball_track.csv"

pose_df = pd.read_csv(pose_csv) if pose_csv.exists() else synthetic_pose_sequence(n=80)
ball_df = pd.read_csv(ball_csv) if ball_csv.exists() else pd.DataFrame()

In [ ]:
angle_png = RESULTS / "d5_02_pose_angles.png"
plot_angle_series(
    pose_df, ["elbow_angle", "knee_angle", "shoulder_angle"], output_path=angle_png
)
print("saved:", angle_png)

In [ ]:
if not ball_df.empty:
    ball_png = RESULTS / "d5_02_ball_path.png"
    plot_ball_path(ball_df, output_path=ball_png)
    print("saved:", ball_png)
else:
    print("沒有 ball_df，略過球軌跡圖。")

In [ ]:
# 選一個 elbow angle 最大的 frame 當展示骨架。
idx = (
    int(pose_df["elbow_angle"].idxmax())
    if "elbow_angle" in pose_df
    else len(pose_df) // 2
)
row = pose_df.iloc[idx]
skeleton = draw_skeleton(640, 480, row)
show_image(skeleton, f"showcase skeleton frame={int(row.frame)}")
skeleton_png = RESULTS / "d5_02_showcase_skeleton.png"
save_image_rgb(skeleton_png, skeleton)
print("saved:", skeleton_png)